# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [ ]:
# Tables Bronze à croiser (écrites au Jour 1)
BRONZE_TABLES: dict[str, str] = {
    "main": "/lakehouse/bronze/steam_reviews",  # Dataset 1 : faits, 1 ligne = 1 avis joueur
    "secondary": "/lakehouse/bronze/steam_games",  # Dataset 2 : dimension, 1 ligne = 1 jeu du catalogue
}

assert len(BRONZE_TABLES) > 0, "TODO : Renseignez vos chemins Delta Bronze dans `BRONZE_TABLES`."
bronze_dfs = {alias: spark.read.format("delta").load(path) for alias, path in BRONZE_TABLES.items()}
for alias, df in bronze_dfs.items():
    print(f"[{alias}] {BRONZE_TABLES[alias]} -> {df.count():,} lignes | colonnes : {df.columns[:8]}")

df_bronze = bronze_dfs.get("main")
df_games_bronze = bronze_dfs["secondary"]
META_COLS = {"_ingested_at", "_source_dataset", "_source_file"}

# Colonnes clés auditées : toutes les colonnes métier des avis + clé de jointure, libellé, axe temporel et
# métriques du catalogue. Un seul agg() par table : 1 seul passage sur les données.
KEY_COLS = {
    "steam_reviews": [c for c in df_bronze.columns if c not in META_COLS],
    "steam_games": ["appid", "name", "release_date", "price", "positive_ratings", "negative_ratings"],
}
frames = {"steam_reviews": df_bronze, "steam_games": df_games_bronze}
report_rows = []
for name, cols in KEY_COLS.items():
    agg = frames[name].agg(
        F.count(F.lit(1)).alias("_n"), *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]
    ).first()
    report_rows += [(name, c, int(agg[c]), round(100 * agg[c] / agg["_n"], 3)) for c in cols]
NULL_SCHEMA = "table string, column string, null_count long, null_pct double"
null_report = spark.createDataFrame(report_rows, NULL_SCHEMA)
assert null_report is not None, "TODO : Calculez le diagnostic de valeurs nulles `null_report`."
null_report.show(truncate=False)

# Doublons : lignes métier strictement identiques (hors métadonnées Bronze) et doublons de clé
for name, df, key in [("steam_reviews", df_bronze, None), ("steam_games", df_games_bronze, "appid")]:
    biz = [c for c in df.columns if c not in META_COLS]
    n_rows = df.count()
    n_dup = n_rows - df.select(biz).distinct().count()
    n_null_all = df.agg(sum(F.sum(F.col(c).isNull().cast("int")) for c in biz).alias("n")).first()["n"]
    msg = f"[{name}] {n_rows:,} lignes | {n_dup:,} doublons stricts ({100 * n_dup / n_rows:.1f} %)"
    msg += f" | {n_null_all:,} NULL sur {len(biz)} colonnes métier"
    if key:
        msg += f" | {n_rows - df.select(key).distinct().count():,} doublon(s) de clé `{key}`"
    print(msg)

df_bronze.groupBy("review_score", "review_votes").count().orderBy("review_score", "review_votes").show()


#### Analyse de l'audit qualité Bronze *(exécution du 2026-11-17)*

| Table | Lignes | NULL | Doublons stricts | Doublons de clé |
|---|---|---|---|---|
| `steam_reviews` (faits) | 3 763 740 | 42 527 sur `app_name` (1,13 %), **0 sur la clé `app_id`** | 3 732 824 (**99,2 %**) | n/a (pas d'identifiant d'avis) |
| `steam_games` (dimension) | 27 075 | 0 sur les 18 colonnes | 0 | **0 sur `appid`** |

- **Clés de jointure saines** : `app_id` n'est jamais NULL côté faits, `appid` est complet et unique côté dimension → relation **N-1** garantie, pas de risque d'explosion N-M.
- **Valeurs du domaine valides** : `review_score ∈ {-1, 1}` et `review_votes ∈ {0, 1}` uniquement (tableau croisé ci-dessus). 82,6 % d'avis positifs et 14,9 % d'avis jugés « utiles » : la variable cible est **déséquilibrée**, à garder en tête pour les tests du Jour 4.
- **`app_name` NULL ou divergent** : 42 527 NULL côté avis, et des libellés qui diffèrent du catalogue (symboles ™/®, renommages). Le libellé de référence sera donc **`name` du catalogue** (`game_name`), et `app_name` est écarté de Silver.
- **Piège des « doublons » à 99,2 %** : la version Kaggle des avis ne garde que 4 colonnes (le texte et l'horodatage ont été écartés). Il ne reste que 30 916 combinaisons distinctes `(app_id, app_name, review_score, review_votes)` pour 3,76 M d'avis : des milliers de joueurs ont recommandé Counter-Strike sans recevoir de vote. Ce sont des **avis distincts et légitimes**, pas des doublons techniques. Un `dropDuplicates()` sur la ligne entière détruirait 99,2 % des faits et fausserait tous les volumes. Les vrais doublons techniques (double ingestion d'un même fichier) sont déjà exclus par l'écriture Bronze en `overwrite`. `dropDuplicates` est donc appliqué **là où il a un sens métier : sur la clé `appid` de la dimension**.


## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).

### Exercice 2.1 — Nettoyer, harmoniser vos clés, joindre vos sources et écrire `/lakehouse/silver/<table_silver>`
1. Nettoyez vos tables (`dropDuplicates`, filtrage d'anomalies, typage fort `F.to_timestamp` / `F.to_date` / numériques) et harmonisez la clé de jointure commune.
2. Créez au moins **2 nouvelles colonnes métier dérivées** (dont idéalement un indicateur combinant vos deux sources !) utiles pour les Jours 3 & 4.
3. Réalisez la jointure (avec `F.broadcast(...)` sur une table de dimension/dataset secondaire compact), contrôlez le **taux de rapprochement (*match rate*)** ainsi que l'absence de duplication N-M, et affichez `.explain("simple")`.
4. Écrivez la table enrichie au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [ ]:
SILVER_MAIN_PATH = "/lakehouse/silver/steam_reviews_enriched"
HOLDOUT_SIZE = 500  # derniers avis retenus comme « lot J+1 » pour le MERGE réaliste de l'exercice 3.2

# Tags placés dans `genres` qui décrivent le contenu ou le modèle économique, pas un genre de jeu
NON_GENRE_TAGS = ["Violent", "Gore", "Nudity", "Sexual Content", "Free to Play", "Early Access"]

# 1) Table principale (avis) : filtres de valeurs impossibles + typage + clé technique.
#    PAS de dropDuplicates() sur la ligne complète : sans identifiant d'avis, deux avis identiques
#    (même jeu, même note, même vote) sont des avis distincts et légitimes (cf. audit ci-dessus).
reviews_clean = (
    df_bronze.filter(F.col("app_id").isNotNull())
    .filter(F.col("review_score").isin(-1, 1))  # recommandé (1) / non recommandé (-1)
    .filter(F.col("review_votes").isin(0, 1))  # avis jugé utile par la communauté (1) ou non (0)
    .select(
        F.monotonically_increasing_id().alias("review_id"),  # clé technique Silver (MERGE)
        F.col("app_id").cast("int").alias("app_id"),
        F.col("review_score").cast("int").alias("review_score"),
        F.col("review_votes").cast("int").alias("review_votes"),
        F.col("_ingested_at").alias("_bronze_ingested_at"),
    )
)
count_before_join = reviews_clean.count()

# 2) Dimension (catalogue) : typage explicite de l'axe temporel, filtres, unicité de la clé de jointure
genre_list = F.array_except(F.split("genres", ";"), F.array(*[F.lit(t) for t in NON_GENRE_TAGS]))
games_clean = (
    df_games_bronze.withColumn(
        "release_date", F.to_date(F.col("release_date").cast("string"), "yyyy-MM-dd")
    )
    .filter(F.col("appid").isNotNull() & F.col("name").isNotNull())
    .filter(F.col("release_date").isNotNull() & (F.col("release_date") <= F.current_date()))
    .filter((F.col("price") >= 0) & (F.col("positive_ratings") >= 0) & (F.col("negative_ratings") >= 0))
    .filter((F.col("average_playtime") >= 0) & (F.col("median_playtime") >= 0))
    .dropDuplicates(["appid"])  # garantit une relation N-1 : aucune explosion possible à la jointure
    .select(
        F.col("appid").cast("int").alias("appid"),
        F.col("name").alias("game_name"),
        "release_date",
        F.year("release_date").alias("release_year"),
        "developer",
        "publisher",
        "platforms",
        "required_age",
        F.coalesce(F.try_element_at(genre_list, F.lit(1)), F.lit("Other")).alias("primary_genre"),
        "genres",
        "categories",
        "steamspy_tags",
        "english",
        "achievements",
        "positive_ratings",
        "negative_ratings",
        F.round(
            F.try_divide(
                F.col("positive_ratings"), F.col("positive_ratings") + F.col("negative_ratings")
            ),
            4,
        ).alias("catalog_positive_ratio"),
        "average_playtime",
        "median_playtime",
        "owners",
        (
            (F.split("owners", "-")[0].cast("long") + F.split("owners", "-")[1].cast("long")) / 2
        ).cast("long").alias("owners_mid"),
        F.col("price").cast("double").alias("price"),
        F.when(F.col("price") == 0, "0. Gratuit")
        .when(F.col("price") < 5, "1. < 5 £")
        .when(F.col("price") < 15, "2. 5-15 £")
        .when(F.col("price") < 30, "3. 15-30 £")
        .otherwise("4. >= 30 £")
        .alias("price_tier"),
    )
)
n_games_bronze, n_games_clean = df_games_bronze.count(), games_clean.count()
n_distinct_appid = games_clean.select("appid").distinct().count()
assert n_distinct_appid == n_games_clean, "Clé appid non unique côté dimension !"

# 3) Jointure N-1 faits -> dimension : la dimension (27 k lignes, ~5 Mo) est diffusée à chaque tâche
df_joined = reviews_clean.join(F.broadcast(games_clean), reviews_clean.app_id == games_clean.appid, "left")
n_matched = df_joined.filter(F.col("appid").isNotNull()).count()
match_rate = n_matched / count_before_join

df_silver = (
    df_joined.filter(F.col("appid").isNotNull())  # les avis orphelins resteraient en Bronze (quarantaine)
    .drop("appid")
    .withColumn("is_recommended", F.col("review_score") == 1)
    .withColumn("is_helpful", F.col("review_votes") == 1)
    # Indicateur inter-sources : l'avis contredit-il le consensus du catalogue (ratio positif >= 50 %) ?
    .withColumn("is_contrarian_review", F.col("is_recommended") != (F.col("catalog_positive_ratio") >= 0.5))
    .withColumn("_processed_at", F.current_timestamp())
)

assert df_silver is not None, "TODO : Construisez le DataFrame nettoyé et joint `df_silver`."
df_silver.explain("simple")

physical_plan = df_silver._jdf.queryExecution().executedPlan().toString()
assert "BroadcastHashJoin" in physical_plan, "La jointure n'est pas un BroadcastHashJoin !"
print("Stratégie physique détectée : BroadcastHashJoin (BuildRight = dimension steam_games diffusée)")
print(f"Avis Bronze -> nettoyés : {df_bronze.count():,} -> {count_before_join:,}")
print(f"Jeux Bronze -> nettoyés : {n_games_bronze:,} -> {n_games_clean:,}")
print(f"Match rate avis -> catalogue : {match_rate:.2%} ({n_matched:,} / {count_before_join:,})")
assert n_matched <= count_before_join, "Explosion N-M détectée à la jointure !"

# Lot J+1 : les HOLDOUT_SIZE plus grands review_id sont retenus pour l'upsert de l'exercice 3.2
holdout_min_id = (
    reviews_clean.select("review_id").orderBy(F.desc("review_id")).limit(HOLDOUT_SIZE)
    .agg(F.min("review_id")).first()[0]
)

# Écriture Delta Silver (chargement initial = tout sauf le lot J+1)
(
    df_silver.filter(F.col("review_id") < holdout_min_id)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_MAIN_PATH)
)

silver_count = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
if count_before_join is not None:
    assert silver_count <= count_before_join, (
        f"Alerte explosion N-M : {silver_count:,} lignes après jointure > {count_before_join:,} lignes avant jointure !"
    )
print(f"Table Silver écrite dans {SILVER_MAIN_PATH} : {silver_count:,} lignes")
print(f"(+ {n_matched - silver_count:,} avis du lot J+1 réservés au MERGE de l'exercice 3.2)")


#### Choix de nettoyage, colonnes dérivées et lecture du plan d'exécution

**Nettoyage** (règles appliquées ; les données Kaggle étant déjà propres, 0 ligne rejetée : 3 763 740 → 3 763 740 avis, 27 075 → 27 075 jeux)
- Avis : clé `app_id` non NULL, `review_score ∈ {-1, 1}`, `review_votes ∈ {0, 1}`, typage `int` explicite.
- Catalogue : `release_date` re-typée explicitement en `date` (`F.to_date(..., "yyyy-MM-dd")`) et non future, `price ≥ 0`, notes et temps de jeu ≥ 0, `dropDuplicates(["appid"])`. `release_date` est le **seul axe temporel** du projet (les avis n'ont pas d'horodatage) : il alimentera la série `LAG()` du Jour 3.
- **Clé technique `review_id`** (`monotonically_increasing_id`) : les avis n'ont pas d'identifiant naturel, or un `MERGE` en exige un. Elle est unique et stable pour une version de Bronze donnée (elle dépend de la partition et du rang dans la partition), mais n'est pas un identifiant métier.

**Colonnes métier dérivées** (pour les Jours 3 et 4)

| Colonne | Source | Définition / usage |
|---|---|---|
| `is_recommended`, `is_helpful` | avis | booléens issus de `review_score = 1` / `review_votes = 1` → taux de satisfaction / d'utilité |
| `release_year` | catalogue | `year(release_date)` → axe temporel (Gold `LAG`) |
| `price_tier` | catalogue | 5 gammes de prix (en £, boutique UK) : Gratuit, < 5, 5-15, 15-30, ≥ 30 ; seuils proches des quantiles P50 = 3,99, P90 = 14,99, P99 = 30,99 |
| `primary_genre` | catalogue | 1er genre réel, une fois retirés les tags de contenu (`Violent`, `Nudity`, `Free to Play`…) |
| `catalog_positive_ratio` | catalogue | `positive / (positive + negative)` via `try_divide` (mode ANSI de Spark 4 : pas d'erreur si dénominateur nul) |
| `owners_mid` | catalogue | milieu de la tranche de propriétaires `"20000-50000"` → 35 000 (variable numérique de popularité) |
| `is_contrarian_review` | **avis × catalogue** | l'avis contredit-il le consensus du jeu (ratio ≥ 50 %) ? Indicateur **inter-sources** |

**Lecture du plan physique (`explain("simple")`)**
- `BroadcastHashJoin [app_id], [appid], Inner, BuildRight` : la dimension (27 k lignes, ~5 Mo) est construite côté droit (*BuildRight*) puis envoyée une fois à chaque tâche (`BroadcastExchange`). Les 3,76 M d'avis sont joints **sur place**, sans aucun `Exchange` (*shuffle*) côté faits : c'est l'intérêt majeur du broadcast. Un `SortMergeJoin` aurait redistribué les deux tables sur le réseau par hachage de la clé.
- Le `LEFT JOIN` suivi du filtre `appid IS NOT NULL` a été **réécrit en `Inner`** par l'optimiseur Catalyst, qui a vu que les lignes non rapprochées étaient de toute façon éliminées. Le *match rate* (100 %) est mesuré avant ce filtre.
- **Narrow vs Wide** : filtres, `select` et `withColumn` sont des transformations *narrow* (chaque partition est traitée seule, au sein d'un même *stage*). Le seul `Exchange hashpartitioning(appid, 200)` est provoqué par le `dropDuplicates` de la petite dimension : c'est une transformation *wide*, d'où une frontière de *stage* visible dans Spark UI.
- **Predicate pushdown** : les filtres des avis apparaissent dans `PushedFilters` du `FileScan parquet`. Ils sont poussés jusqu'au lecteur Parquet, qui peut sauter des *row groups* grâce aux statistiques min/max.
- **Lazy evaluation** : rien n'est calculé à la définition de `df_silver`. Seules les *actions* (`count`, `first`, `save`) déclenchent des jobs, d'où plusieurs jobs successifs dans Spark UI pour une seule chaîne de transformations.

**Résultat** : `/lakehouse/silver/steam_reviews_enriched`, 31 colonnes, 8 fichiers Parquet. Le chargement initial contient 3 763 240 avis ; les 500 derniers sont volontairement retenus comme « lot J+1 » pour l'exercice 3.2.


## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [ ]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# Structure du commit : 1 action par ligne JSON (commitInfo, protocol, metaData, add = 1 fichier Parquet)
log0 = spark.read.json(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json")
print("Actions du commit 0 :", log0.columns)
print("Fichiers Parquet ajoutés (actions add) :", log0.filter(F.col("add").isNotNull()).count())
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
log_dir = jvm.org.apache.hadoop.fs.Path(f"{SILVER_MAIN_PATH}/_delta_log")
log_files = [st.getPath().getName() for st in fs.listStatus(log_dir)]
print("Contenu de _delta_log :", sorted(log_files))

# 2. Schema Enforcement : un DataFrame au schéma incompatible doit être rejeté
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
schema_blocked = False
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    schema_blocked = True
    print("Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

assert schema_blocked, "Delta aurait dû refuser l'append d'un schéma incompatible !"
n_after = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
assert n_after == silver_count, "L'écriture refusée ne doit laisser aucune ligne partielle (atomicité)"
print(f"Table intacte après l'échec : {n_after:,} lignes (aucune écriture partielle)")


#### Analyse : journal `_delta_log` et Schema Enforcement

- Le commit `00000000000000000000.json` est un fichier **JSON Lines** : une action par ligne. `commitInfo` (opération `WRITE`, mode `Overwrite`, métriques), `protocol` (versions minimales lecteur/écrivain), `metaData` (identifiant de table et **schéma** sérialisé) et **8 actions `add`**, une par fichier Parquet écrit, avec ses statistiques min/max. Une table Delta est donc *un dossier de Parquet + un journal ordonné qui dit quels fichiers sont valides* : c'est ce journal qui apporte l'atomicité sur HDFS, où l'écriture de 8 fichiers n'est pas atomique en soi. Le fichier `.crc` est un résumé de contrôle de la version (taille, nombre de fichiers).
- **Schema Enforcement** : l'`append` d'un DataFrame `(fake_id, intruder_col)` est rejeté par `[DELTA_METADATA_MISMATCH]`. Delta compare le schéma entrant au `metaData` du journal **avant** de valider le commit. Aucun fichier n'est référencé et la table reste à 3 763 240 lignes (vérifié par `assert`) : c'est l'atomicité. Un *data lake* Parquet brut aurait accepté ces fichiers et corrompu silencieusement les lectures.
- Une évolution de schéma **voulue** se déclare explicitement (`.option("mergeSchema", "true")` ou `overwriteSchema` comme en Bronze) : le choix devient une décision tracée dans le journal, pas un accident.


### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [ ]:
# MERGE INTO réaliste : arrivée du lot J+1 = 500 nouveaux avis (insert) + 500 avis déjà chargés
# re-livrés par la source (update idempotent, sans doublon). Puis comparaison des versions.
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
version_before_merge = target_delta.history(1).select("version").first()[0]

new_rows = df_silver.filter(F.col("review_id") >= holdout_min_id)
redelivered_rows = (
    df_silver.filter(F.col("review_id") < holdout_min_id).orderBy(F.desc("review_id")).limit(HOLDOUT_SIZE)
)
# Lot matérialisé (1 000 lignes) : la source du MERGE est figée et indépendante de la table cible
batch_rows = (
    new_rows.unionByName(redelivered_rows).withColumn("_processed_at", F.current_timestamp()).collect()
)
batch_df = spark.createDataFrame(batch_rows, df_silver.schema)

(
    target_delta.alias("t")
    .merge(batch_df.alias("s"), "t.review_id = s.review_id")
    .whenMatchedUpdate(
        set={
            "review_votes": "s.review_votes",
            "is_helpful": "s.is_helpful",
            "_processed_at": "s._processed_at",
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)
assert history_df.count() >= 2, "TODO : Exécutez au moins une opération MERGE/UPDATE pour créer une version >= 1 !"

# Time Travel : version initiale (0), version juste avant ce MERGE, version courante
last_merge = history_df.filter("operation = 'MERGE'").orderBy(F.desc("version")).first()
merge_metrics = last_merge["operationMetrics"]
print("MERGE -> insérées :", merge_metrics["numTargetRowsInserted"], "| mises à jour :",
      merge_metrics["numTargetRowsUpdated"])
v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_MAIN_PATH)
v_before = spark.read.format("delta").option("versionAsOf", version_before_merge).load(SILVER_MAIN_PATH)
v_now = spark.read.format("delta").load(SILVER_MAIN_PATH)
n_v0, n_before, n_now = v0.count(), v_before.count(), v_now.count()
print(f"versionAsOf 0 : {n_v0:,} lignes | version {version_before_merge} (avant MERGE) : {n_before:,} "
      f"| version courante : {n_now:,}")

assert n_now == n_before + HOLDOUT_SIZE, "Le MERGE doit insérer exactement le lot de nouveaux avis"
assert n_now == n_matched, "Après le lot J+1, Silver doit contenir tous les avis rapprochés"
n_distinct_ids = v_now.select("review_id").distinct().count()
assert n_distinct_ids == n_now, "Les avis re-livrés ne doivent pas être dupliqués"

# Les avis re-livrés : même contenu, seul _processed_at a changé entre les deux versions
sample_ids = [r["review_id"] for r in redelivered_rows.select("review_id").limit(3).collect()]
(
    v_before.filter(F.col("review_id").isin(sample_ids)).select("review_id", "game_name", "_processed_at")
    .withColumnRenamed("_processed_at", "processed_at_avant")
    .join(v_now.select("review_id", F.col("_processed_at").alias("processed_at_apres")), "review_id")
    .show(truncate=False)
)


#### Analyse : `MERGE INTO` et Time Travel

**Scénario** : chaque jour, la source livre un lot d'avis. Celui de J+1 contient **500 nouveaux avis** et **500 avis déjà chargés, re-livrés** (cas fréquent : la source renvoie une fenêtre glissante). Un `append` naïf aurait créé 500 doublons. Le `MERGE` sur la clé `review_id` rend l'ingestion **idempotente** :
- `whenNotMatchedInsertAll` → **500 lignes insérées** (`numTargetRowsInserted`) ;
- `whenMatchedUpdate` → **500 lignes mises à jour** (`numTargetRowsUpdated`) : seuls `review_votes`, `is_helpful` et la date de traitement `_processed_at` sont rafraîchis, sans dupliquer l'avis (unicité de `review_id` vérifiée par `assert`) ;
- au final, Silver contient exactement les **3 763 740** avis rapprochés.

**Coût du MERGE (copy-on-write)** : `numTargetRowsCopied = 481 627`. Parquet est immuable : Delta **réécrit entièrement** le fichier qui contient les lignes modifiées (les 500 avis re-livrés sont tous dans le dernier fichier). Le commit de la version 1 contient **1 action `remove`** (l'ancien fichier) et **8 actions `add`** : 482 627 lignes = 481 627 recopiées + 500 mises à jour + 500 insérées, réparties sur 8 tâches. Les 7 autres fichiers restent intacts. Revers de la médaille : chaque MERGE ajoute de petits fichiers, d'où l'intérêt de `OPTIMIZE` au Jour 3.

**Time Travel** : `DESCRIBE HISTORY` trace les versions `0 = WRITE` et `1 = MERGE`. `versionAsOf 0` relit l'état initial (**3 763 240** lignes), alors que la version courante en compte **3 763 740**. Pour les avis re-livrés, `_processed_at` montre l'horodatage de la version 0 à côté de celui de la version 1. C'est possible parce que les fichiers `remove` ne sont pas effacés physiquement tant qu'un `VACUUM` n'est pas lancé (7 jours de rétention par défaut). Cela permet l'audit (« qu'affichait le dashboard hier ? »), la reproductibilité d'une analyse et le *rollback* (`RESTORE TABLE ... TO VERSION AS OF 0`).

> Ré-exécuter le notebook ajoute de nouvelles versions (`WRITE` en overwrite, puis `MERGE`) : l'historique s'allonge, mais le contenu courant reste identique (3 763 740 avis). `versionAsOf 0` désigne toujours le tout premier chargement.
